# 패키지 불러오기 및 데이터 불러오기

In [57]:
# 사용자 정의 모듈
from hossam import load_data, my_dpi

# 데이터 처리
import numpy as np
from pandas import DataFrame

# 시각화
import seaborn as sb
from matplotlib import pyplot as plt

# 머신러닝
from sklearn.preprocessing import StandardScaler
from sklearn.cluster._dbscan import DBSCAN
from sklearn.neighbors import NearestNeighbors
from kneed import KneeLocator

# 수학 / 기하
from scipy.spatial import ConvexHull


In [58]:
from sklearn.preprocessing import MinMaxScaler

from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np

from scipy.spatial import ConvexHull
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.spatial import ConvexHull

my_dpi = 200


In [59]:
my_dpi = 200 # 이미지 선명도(100~300)
fpath = "./NotoSansKR-Regular.ttf" # 한글을 지원하는 폰트 파일의 경로
fm.fontManager.addfont(fpath) # 폰트의 글꼴을 시스템에 등록함
fprop = fm.FontProperties(fname=fpath) # 폰트의 속성을 읽어옴
fname = fprop.get_name() # 읽어온 속성에서 폰트 이름 추출
plt.rcParams['font.family'] = fname # 그래프에 한글 폰트 적용
plt.rcParams['font.size'] = 6 # 기본 폰트 크기
plt.rcParams['axes.unicode_minus'] = False # 그래프에 마이너스 깨짐 방지

In [60]:
origin = load_data("game_usage")


게임 이용시간(time spent)과 레벨(game level)에 대한 가상 데이터


### 데이터 전처리

In [66]:
scaler = StandardScaler()
df = DataFrame(scaler.fit_transform(origin), columns=origin.columns)
df.head()

,time spent,game level
0,-0.250733,1.474805
1,0.326494,0.606546
2,-0.611500,0.795456
3,0.470801,1.674613
4,-1.405187,-1.558652


In [67]:
min_samples=3

In [73]:
# -------------------------------
# k = min_samples 설정
# -------------------------------
k = min_samples


# -------------------------------
# 각 점에 대해 k번째 최근접 이웃 거리 계산
# -------------------------------
neighbors = NearestNeighbors(
    n_neighbors=k
)
neighbors_fit = neighbors.fit(df)

distance, indices = neighbors_fit.kneighbors(df)


# -------------------------------
# 모든 점의 거리 값을 가까운 순서대로 정렬 (오름차순)
# -------------------------------
s_distance = np.sort(
    distance,
    axis=0
)


# -------------------------------
# 각 데이터 포인트로부터의 k번째 거리 추출
# -------------------------------
target = s_distance[:, k - 1]


# -------------------------------
# 엘보우 포인트 탐색
# -------------------------------
kl = KneeLocator(
    range(0, len(target)),
    target,
    curve="convex",
    direction="increasing"
)

eps = kl.elbow_y


# -------------------------------
# 결과 출력
# -------------------------------
print(f"eps: {eps}")


eps: 0.41251429498079606


### eps 값,구간찾기

In [74]:
delta_ratio = 0.3
step_ratio = 0.05

eps_min = eps * (1-delta_ratio)
eps_max = eps * (1 + delta_ratio)
step = eps*step_ratio

eps_grid = np.arange(eps_min,eps_max+step,step)
eps_grid

array([0.28876001, 0.30938572, 0.33001144, 0.35063715, 0.37126287,
       0.39188858, 0.41251429, 0.43314001, 0.45376572, 0.47439144,
       0.49501715, 0.51564287, 0.53626858, 0.5568943 ])

### eps별 DBSCAN 결과 얻기

In [80]:
labels_dict = {}
for eps in eps_grid:
    estimator = DBSCAN(eps=eps, mins_samples=min_samples)
    labels_dict[eps]=estimator.fit_predict(df)

labels_dict

TypeError: DBSCAN.__init__() got an unexpected keyword argument 'mins_samples'. Did you mean 'min_samples'?

### eps별 기본 지표

In [ ]:
rows = []

for eps in eps_grid:

    labels = labels_dict[eps]

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = np.mean(labels == -1)

    rows.append(
        {
            "eps": round(eps, 4),
            "n_clusters": n_clusters,
            "noise_ratio": round(noise_ratio, 4),
        }
    )


edf = DataFrame(rows)
edf


NameError: name 'eps_grid' is not defined

### 이전 구간 대비 ARI값 추가

In [ ]:
ari_list = [np.nan]

for i in range(1, len(eps_grid)):

    ari = adjusted_rand_score(
        labels_dict[eps_grid[i - 1]],
        labels_dict[eps_grid[i]]
    )

    ari_list.append(round(ari, 4))


edf["ARI"] = ari_list
edf


NameError: name 'eps_grid' is not defined

In [ ]:
# -------------------------------
# 안정성 기준 설정
# -------------------------------
ari_threshold = 0.9          # 안정성 ARI 기준 (기본 0.9)
noise_diff_threshold = 0.05  # 이상치 비율 변화 허용치 (기본 0.05)


# -------------------------------
# 변화량 계산
# -------------------------------
edf["cluster_diff"] = edf["n_clusters"].diff().abs()
edf["noise_diff"] = edf["noise_ratio"].diff().abs()


# -------------------------------
# 안정 조건 정의
# -------------------------------
edf["stable"] = (
    (edf["ARI"] >= ari_threshold)
    & (edf["cluster_diff"] == 0)
    & (edf["noise_diff"] <= noise_diff_threshold)
)


# -------------------------------
# 첫 행은 비교 불가 → False 처리
# -------------------------------
edf.loc[edf.index[0], "stable"] = False


# 결과 확인
edf


NameError: name 'edf' is not defined

## ✏️ 인사이트

- **`stable = True`가 연속적으로 유지되는 구간**을  
  최종 **`eps` 후보 범위**로 선정한다.
  - ❌ 단일 `eps` 선택  
  - ⭕ **구간 기반 선택**

- **`eps_grid` 전반에서**
  - ARI 안정성
  - 군집 수 유지
  - 이상치 비율의 완만한 변화  
  를 **동시에 만족하는 구간**을  
  **안정 구간(stable region)**으로 정의하였다.

---

### 🔎 해석 포인트

- DBSCAN 파라미터 선택은 **점(point)이 아니라 구간(region)** 문제이다.
- 군집 구조가 급격히 변하지 않는 구간이  
  **재현성(reproducibility)**과 **일반화 성능** 측면에서 가장 안정적이다.
- ARI + 군집 수 + noise 비율을 함께 고려하는 방식은  
  **단일 지표 기반 선택보다 훨씬 견고한 접근**이다.


### eps 안정구간 도출

In [ ]:
# -------------------------------
# stable 여부를 0/1로 변환
# -------------------------------
stable_flag = edf["stable"].astype(int).values


# -------------------------------
# 연속 구간 그룹 ID 생성
# -------------------------------
group_id = (stable_flag != np.roll(stable_flag, 1)).cumsum()
edf["group_id"] = group_id


# -------------------------------
# stable == True 인 구간만 필터
# -------------------------------
stable_groups = (
    edf[edf["stable"]]
    .groupby("group_id")
)


# -------------------------------
# 각 구간의 길이 계산
# -------------------------------
group_sizes = stable_groups.size()


# -------------------------------
# 가장 긴 안정 구간 선택
# -------------------------------
best_group_id = group_sizes.idxmax()


# -------------------------------
# 해당 구간의 eps 리스트 추출
# -------------------------------
stable_eps_list = (
    edf.loc[
        edf["group_id"] == best_group_id,
        "eps"
    ]
    .tolist()
)

stable_eps_list


NameError: name 'edf' is not defined

### eps구간별 군집결과 확인

In [71]:
for eps in stable_eps_list:

    # ===============================
    # DBSCAN 군집화
    # ===============================
    estimator = DBSCAN(
        eps=eps,
        min_samples=min_samples
    )

    estimator.fit(df)

    result_df = df.copy()
    result_df["cluster"] = estimator.labels_


    # ===============================
    # 벡터 유형 구분
    # ===============================
    result_df["vector"] = "border"

    result_df.loc[
        estimator.core_sample_indices_,
        "vector"
    ] = "core"

    result_df.loc[
        result_df["cluster"] == -1,
        "vector"
    ] = "noise"


    # ===============================
    # 시각화 데이터 설정
    # ===============================
    vdf = result_df.copy()

    hue_field = "cluster"
    x_field = "time spent"
    y_field = "game level"


    # ===============================
    # 그래프 초기화
    # ===============================
    figsize = (1280 / my_dpi, 720 / my_dpi)
    fig, ax = plt.subplots(
        nrows=1,
        ncols=1,
        figsize=figsize,
        dpi=my_dpi
    )


    # ===============================
    # 군집별 Convex Hull (noise 제외)
    # ===============================
    for c in vdf[hue_field].unique():

        # 이상치는 제외
        if c == -1:
            continue

        df_c = vdf.loc[
            vdf[hue_field] == c,
            [x_field, y_field]
        ]

        try:
            hull = ConvexHull(df_c)

            points = np.append(
                hull.vertices,
                hull.vertices[0]
            )

            ax.plot(
                df_c.iloc[points, 0],
                df_c.iloc[points, 1],
                linewidth=0.5,
                linestyle=":"
            )

            ax.fill(
                df_c.iloc[points, 0],
                df_c.iloc[points, 1],
                alpha=0.1
            )

        except Exception:
            pass


    # ===============================
    # 핵심(core) 벡터 표시
    # ===============================
    sb.scatterplot(
        data=vdf[
            (vdf[hue_field] != -1) &
            (vdf["vector"] == "core")
        ],
        x=x_field,
        y=y_field,
        hue=hue_field,
        edgecolor="white",
        linewidth=0.8
    )


    # ===============================
    # 외곽(border) 벡터 표시
    # ===============================
    sb.scatterplot(
        data=vdf[
            (vdf[hue_field] != -1) &
            (vdf["vector"] == "border")
        ],
        x=x_field,
        y=y_field,
        hue=hue_field,
        marker="^",
        s=25,
        edgecolor="#888",
        linewidth=0.8
    )


    # ===============================
    # 노이즈(noise) 벡터 표시
    # ===============================
    sb.scatterplot(
        data=vdf[vdf["vector"] == "noise"],
        x=x_field,
        y=y_field,
        color="red",
        marker="x",
        s=70
    )


    # ===============================
    # 그래프 마무리
    # ===============================
    ax.set_title(
        f"DBSCAN Clustering (min_samples={min_samples}, eps={round(eps, 4)})",
        fontsize=8
    )

    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()


NameError: name 'stable_eps_list' is not defined

In [ ]:
from sklearn.neighbors import NearestNeighbors
print("NearestNeighbors OK")

NearestNeighbors OK


In [ ]:
import sklearn
import pkgutil

print("sklearn subpackages:")
for m in pkgutil.iter_modules(sklearn.__path__):
    print(m.name)


sklearn subpackages:
__check_build
_build_utils
_config
_cyutility
_distributor_init
_isotonic
_loss
_min_dependencies
base
calibration
cluster
compose
conftest
covariance
cross_decomposition
datasets
decomposition
discriminant_analysis
dummy
ensemble
exceptions
experimental
externals
feature_extraction
feature_selection
frozen
gaussian_process
impute
inspection
isotonic
kernel_approximation
kernel_ridge
linear_model
manifold
metrics
mixture
model_selection
multiclass
multioutput
naive_bayes
neighbors
neural_network
pipeline
preprocessing
random_projection
semi_supervised
svm
tests
tree
utils
